# 实验7：基于 MLIR 定义 ASC-IR

## 学习目标

1. 理解 MLIR Dialect / Operation / Type 与 ASC-IR 的关系
2. 掌握 TableGen class / defm / multiclass 基本用法
3. 解释 `defm Sub : BinaryTemplateL0123Op<...>` 一行展开 L0~L3
4. 使用 FileCheck 验证 IR 定义

## 环境准备

In [ ]:
cd ~/pyasc
!echo "=== IR 定义文件 ==="
ls include/ascir/Dialect/Asc/IR/Dialect.td
ls include/ascir/Dialect/Asc/IR/Base.td
ls include/ascir/Dialect/Asc/IR/Basic/OpVecBinary.td
!echo ""
!echo "=== 测试文件 ==="
ls test/Target/AscendC/basic/vec_binary.mlir
!echo ""
!echo "=== 所有基本运算 IR ==="
ls include/ascir/Dialect/Asc/IR/Basic/

## 1. ASC-IR 文件组织结构

PyAsc 的 IR 定义在 `include/ascir/Dialect/Asc/IR/`：
- **Dialect.td**：ASC-IR Dialect 入口
- **Base.td**：通用 Op 模板基类（BinaryOp/L0~L3 继承体系）
- **Basic/**：各具体运算的 Operation（OpVecBinary.td 等 30+ 个 .td 文件）
- **Core/**：Type、Attribute、Interface 定义
- **Adv/**：高级运算（Matmul、Activation 等）

## 2. BinaryOp 继承体系

In [ ]:
!grep -n "class Binary\|BinaryL0Op\|BinaryL1Op\|BinaryL2Op\|BinaryL3Op\|BinaryTemplateL0123Op" include/ascir/Dialect/Asc/IR/Base.td

**继承体系（Base.td 定义）：**
- **BinaryL0Op**：连续 mask（int mask + repeat_times）
- **BinaryL1Op**：逐bit mask 列表（List[int] mask）
- **BinaryL2Op**：count 模式（元素个数）
- **BinaryL3Op**：运算符重载（operator+/operator-）
- **BinaryTemplateL0123Op**：multiclass，一次生成 L0~L3 全部 Operation

> **思考：** 为什么分四类而不是统一 BinaryOp？

## 3. Sub 的 TableGen 定义

In [ ]:
!grep -n "defm Sub\|defm Add\|defm Mul\|defm Div" include/ascir/Dialect/Asc/IR/Basic/OpVecBinary.td

**一行展开四种 Operation：**
```
defm Sub : BinaryTemplateL0123Op<"sub", "Sub", "operator-">;
```
三个参数的含义：
- `"sub"` -> MLIR mnemonic 前缀，生成 `ascendc.sub_l0` ~ `ascendc.sub_l3`
- `"Sub"` -> 对应 AscendC API 名 `AscendC::Sub(...)`
- `"operator-"` -> L3 运算符形式 `v1 = v2.operator-(v3);`

`defm` + `multiclass` = TableGen 的宏生成机制。

## 4. FileCheck 测试验证

In [ ]:
# 查看 Sub 的期望 Ascend C 输出
!grep -A2 "ascendc.sub_l" test/Target/AscendC/basic/vec_binary.mlir

**IR -> Ascend C 对应：**
- `ascendc.sub_l0` -> `AscendC::Sub<float, 0>(v1, v2, v3, v4, v4, v5);`
- `ascendc.sub_l1` -> `AscendC::Sub<float, 0>(v1, v2, v3, mask_list, v4, v5);`
- `ascendc.sub_l2` -> `AscendC::Sub(v1, v2, v3, v4);`
- `ascendc.sub_l3` -> `v1 = v2.operator-(v3);`

> L0/L1 带 `<float, 0>` 模板参数（元素类型 + repeat_times），L2 不带，L3 走运算符。

## 5. 任务拓展

选 `mul` 或 `div`：
1. 找其 defm 定义
2. 分析 L0~L3 展开
3. FileCheck 期望输出
4. 比较 operator* / operator/ 与 operator- 的差异

## 总结

1. ASC-IR 基于 MLIR Dialect + TableGen 声明式定义
2. BinaryTemplateL0123Op multiclass 一次生成 L0~L3
3. Sub 一行 defm 展开四类 Operation
4. FileCheck 验证 IR -> Ascend C 翻译正确性